In [1]:
!pip install nltk scikit-learn

In [2]:
import nltk
import re
import string
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [4]:
data = {
    "text": [
        "I love this product",
        "This is the worst experience",
        "Amazing quality and fast delivery",
        "I hate this item",
        "Very satisfied with the purchase",
        "Terrible service",
        "Not bad, could be better",
        "Absolutely fantastic",
        "Worst ever",
        "Pretty good overall"
    ],
    "label": [1,0,1,0,1,0,1,1,0,1]  # 1 = Positive, 0 = Negative
}

df = pd.DataFrame(data)
df.head()

,text,label
0,I love this product,1
1,This is the worst experience,0
2,Amazing quality and fast delivery,1
3,I hate this item,0
4,Very satisfied with the purchase,1


In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    words = text.split()

    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess)
df.head()

,text,label,clean_text
0,I love this product,1,love product
1,This is the worst experience,0,worst experience
2,Amazing quality and fast delivery,1,amazing quality fast delivery
3,I hate this item,0,hate item
4,Very satisfied with the purchase,1,satisfied purchase


In [6]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [8]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [9]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

Accuracy: 0.0

Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       2.0
           1       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [10]:
def predict_sentiment(text):
    text = preprocess(text)
    text_tfidf = vectorizer.transform([text])
    prediction = model.predict(text_tfidf)[0]

    return "Positive 😊" if prediction == 1 else "Negative 😡"

# Try examples
print(predict_sentiment("I really loved this!"))
print(predict_sentiment("This is horrible"))

Positive 😊
Positive 😊
